# Householder QR, from scratch

An interactive, self-paced tour of the algorithm at the heart of this repo:
**batched compact Householder QR factorization**, the thing `torch.geqrf`
computes and the thing your kernels in `kernels/` are racing to compute faster.

By the end you'll be able to answer:

1. What does *QR decomposition* actually compute, and why $A = QR$?
2. What is a **Householder reflection** and why is it the workhorse of stable QR?
3. How do you build the reflector vector $v$ and coefficient $\tau$ (the exact
   formula in `qr_practice/householder.py`)?
4. How does zeroing one column at a time turn $A$ into upper-triangular $R$?
5. What is the **compact storage layout** that `geqrf` returns, and how do you
   rebuild $Q$ from it?
6. Why do GPUs prefer the **blocked / compact-WY** form $I - VTV^\top$?

**How to use this notebook:** run each cell top-to-bottom (`Shift+Enter`).
Cells build on each other. Read the prose first, then run the code and compare
it to what the prose claimed. The 🔬 cells at the end are yours to break.

Launch it with:

```bash
uv sync --group learn
uv run --group learn jupyter lab learn_householder_qr.ipynb
```

## Setup

We only need `numpy`, `torch`, and `matplotlib`. We also import the repo's own
reference implementation from `qr_practice/householder.py` so we can check our
hand-written version against the one your kernels are modeled on.

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `qr_practice` resolves when running from anywhere.
repo_root = Path.cwd()
if not (repo_root / "qr_practice").exists() and (repo_root.parent / "qr_practice").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import torch
import matplotlib.pyplot as plt

from qr_practice.householder import (
    compact_householder_qr,
    blocked_compact_householder_qr,
)

torch.set_printoptions(precision=3, sci_mode=False)
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0)

# We work in float64 throughout so numerical noise never distracts from the math.
DTYPE = torch.float64
print("torch", torch.__version__)

In [ ]:
def show_matrix(M, title="", ax=None, cmap="RdBu_r", annot=True, vlim=None):
    """Heatmap of a small matrix with the numeric value drawn in each cell."""
    M = np.asarray(M, dtype=float)
    if ax is None:
        _, ax = plt.subplots(figsize=(0.7 * M.shape[1] + 1, 0.7 * M.shape[0] + 1))
    if vlim is None:
        vlim = np.abs(M).max() or 1.0
    ax.imshow(M, cmap=cmap, vmin=-vlim, vmax=vlim)
    if annot:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                v = M[i, j]
                ax.text(
                    j, i, f"{v:.2f}",
                    ha="center", va="center",
                    color="black" if abs(v) < 0.6 * vlim else "white",
                    fontsize=9,
                )
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))
    ax.set_title(title, fontsize=11)
    return ax


def show_row(mats, titles, **kw):
    """Show several matrices side by side."""
    n = len(mats)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n + 1, 2.8))
    if n == 1:
        axes = [axes]
    vlim = max((np.abs(np.asarray(m, dtype=float)).max() for m in mats), default=1.0)
    for ax, m, t in zip(axes, mats, titles):
        show_matrix(m, t, ax=ax, vlim=vlim, **kw)
    fig.tight_layout()
    plt.show()


def qr_errors(A, H, tau):
    """Reconstruction + orthogonality error of a compact (H, tau) result.

    This is how correctness is *actually* judged in this repo (see
    tests/test_householder.py and local_eval.py): a factorization is correct if
    Q @ R == A and Q is orthogonal -- NOT if H matches some reference
    bit-for-bit, because QR is only unique up to the signs of R's rows.
    """
    Ab, Hb, tb = (A, H, tau) if A.ndim == 3 else (A[None], H[None], tau[None])
    Q = torch.linalg.householder_product(Hb, tb)   # rebuild Q from compact form
    R = torch.triu(Hb)
    recon = (Q @ R - Ab).abs().max().item()
    eye = torch.eye(Ab.shape[-1], dtype=Ab.dtype).expand_as(Q)
    orth = (Q.transpose(-1, -2) @ Q - eye).abs().max().item()
    return recon, orth

## 1. What QR decomposition computes

Given a square matrix $A$, QR factorization writes it as

$$A = Q\,R$$

where

- $Q$ is **orthogonal**: $Q^\top Q = I$ (its columns are unit vectors, mutually
  perpendicular). Multiplying by $Q$ rotates/reflects but never stretches, so it
  never amplifies rounding error — this is why QR is *numerically stable*.
- $R$ is **upper triangular**: every entry below the diagonal is zero.

It's the matrix version of "rotate your coordinate frame so the problem becomes
triangular, which is easy to solve." It powers least squares, the QR
eigenvalue algorithm, Gram–Schmidt orthonormalization, and more.

Let's look at a concrete $4\times4$ example using PyTorch's built-in QR first,
just to see the shapes.

In [ ]:
A = torch.randn(4, 4, dtype=DTYPE)
Q, R = torch.linalg.qr(A)

show_row([A, Q, R], ["A", "Q  (orthogonal)", "R  (upper triangular)"])

print("Q^T Q == I ? ", torch.allclose(Q.T @ Q, torch.eye(4, dtype=DTYPE), atol=1e-10))
print("Q @ R == A ? ", torch.allclose(Q @ R, A, atol=1e-10))
print("\nNotice R's strictly-lower triangle is all zeros:")
print(R)

## 2. The strategy: orthogonal triangularization

How do we *build* $Q$ and $R$? The Householder idea is beautifully direct:

> Repeatedly multiply $A$ on the left by carefully chosen orthogonal matrices
> $H_1, H_2, \dots$ that each knock the below-diagonal entries of one column to
> zero. After $n{-}1$ such steps the matrix is upper triangular — that's $R$.

$$\underbrace{H_{n-1}\cdots H_2 H_1}_{Q^\top}\; A = R
\quad\Longrightarrow\quad A = \underbrace{H_1 H_2 \cdots H_{n-1}}_{Q}\, R$$

(Each $H_k$ is orthogonal and symmetric, and a product of orthogonals is
orthogonal, so $Q$ comes out orthogonal for free.)

The only ingredient we need is a cheap orthogonal matrix that can zero out the
part of a column below a chosen position. That ingredient is the **Householder
reflection**.

## 3. Householder reflections — the geometry

A Householder reflection mirrors space across a hyperplane through the origin.
Pick a (nonzero) vector $v$ defining the plane's normal; the reflection is

$$H = I - \tau\, v\,v^\top, \qquad \tau = \frac{2}{v^\top v}.$$

Key facts you can verify by hand:

- $H$ is **symmetric** and **orthogonal** ($H^\top H = I$), so $H^{-1} = H$.
- $H$ leaves anything perpendicular to $v$ untouched, and flips the component
  along $v$.

The trick for QR: given a column $x$, we can choose $v$ so that $H$ reflects $x$
**exactly onto the first coordinate axis**:

$$Hx = \beta\, e_1 = (\beta, 0, 0, \dots, 0)^\top.$$

All the entries below the first one become zero — exactly what triangularization
needs. The picture below shows this in 2D: we reflect $x$ across the dashed
mirror line so it lands on the $x$-axis, with the *same length*.

In [ ]:
def reflect(x, v, tau):
    x = np.asarray(x, dtype=float)
    v = np.asarray(v, dtype=float)
    return x - tau * (v @ x) * v

x = np.array([3.0, 2.0])
beta = -np.sign(x[0]) * np.linalg.norm(x)   # land on -x direction (sign chosen for stability, see sec 4)
target = np.array([beta, 0.0])
v = x - target
tau = 2.0 / (v @ v)
Hx = reflect(x, v, tau)

fig, ax = plt.subplots(figsize=(5, 5))
ax.axhline(0, color="0.7", lw=1); ax.axvline(0, color="0.7", lw=1)
# mirror line is perpendicular to v, through origin
d = np.array([-v[1], v[0]]); d = d / np.linalg.norm(d)
ts = np.array([-4, 4])
ax.plot(d[0]*ts, d[1]*ts, "k--", lw=1, label="mirror plane (⊥ v)")
ax.annotate("", xy=x, xytext=(0,0), arrowprops=dict(color="C0", width=2))
ax.annotate("", xy=Hx, xytext=(0,0), arrowprops=dict(color="C3", width=2))
ax.annotate("", xy=v*0.5, xytext=(0,0), arrowprops=dict(color="C2", width=1.5))
ax.text(*x, "  x", color="C0", fontsize=12)
ax.text(*Hx, "  Hx = β·e₁", color="C3", fontsize=12)
ax.text(*(v*0.5), "  v", color="C2", fontsize=12)
ax.set_aspect("equal"); ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.legend(loc="upper right"); ax.set_title("Reflecting x onto the first axis")
plt.show()

print("||x||  =", np.linalg.norm(x))
print("||Hx|| =", np.linalg.norm(Hx), " (length preserved)")
print("Hx     =", Hx, " (second component is ~0)")

## 4. Building the reflector: $\beta$, $\tau$, and the compact $v$

Now the exact recipe — this is precisely `_householder_vector` in
`qr_practice/householder.py`. Given a column $x = (\alpha, x_2, \dots, x_m)$:

1. $\beta = -\operatorname{sign}(\alpha)\,\lVert x\rVert$.
   The sign is chosen **opposite** to $\alpha$ on purpose: it makes the leading
   entry of $v = x - \beta e_1$ a *sum* of same-sign magnitudes instead of a
   difference, avoiding catastrophic cancellation. This is the single most
   important numerical detail in the whole algorithm.
2. $v = x - \beta e_1$, then **normalize so $v_1 = 1$**. Dividing through by
   $v_1 = \alpha - \beta$ lets us store the implicit `1` for free and keep only
   the *tail* $v_{2:}$ — this is the "compact" in compact Householder, and it's
   why `geqrf` can pack everything below the diagonal.
3. $\tau = (\beta - \alpha)/\beta$. With $v_1 = 1$ this equals $2/(v^\top v)$,
   the reflection coefficient.

Edge case: if $x$ below the diagonal is already zero (norm makes $v=0$), set
$\tau = 0$ so $H = I$ (a no-op). The repo handles this with the `zero` /
`denom_safe` masks so it stays vectorized over the whole batch.

In [ ]:
def householder_vector(x):
    """Compact reflector for a 1-D vector x. Mirrors qr_practice._householder_vector.

    Returns (beta, tau, v_tail) where v = [1, *v_tail] and H = I - tau v v^T.
    """
    x = np.asarray(x, dtype=float)
    alpha = x[0]
    norm = np.linalg.norm(x)
    if norm == 0.0:
        return alpha, 0.0, np.zeros_like(x[1:])
    sign = 1.0 if alpha >= 0 else -1.0
    beta = -sign * norm
    v_tail = x[1:] / (alpha - beta)
    tau = (beta - alpha) / beta
    return beta, tau, v_tail


x = np.array([3.0, 1.0, -2.0, 4.0])
beta, tau, v_tail = householder_vector(x)
v = np.concatenate([[1.0], v_tail])

H = np.eye(len(x)) - tau * np.outer(v, v)
print("beta =", round(beta, 4), "  tau =", round(tau, 4))
print("v    =", v)
print("\nH @ x =", H @ x, "  <- should be (beta, 0, 0, 0)")
print("H orthogonal? ", np.allclose(H @ H.T, np.eye(len(x))))

## 5. Zeroing one column of a real matrix

Apply the reflector built from $A$'s first column to the *whole* matrix
($H A$). The first column collapses to $(\beta, 0, \dots, 0)$, and the other
columns get mixed but stay the same size. Watch the first column below.

In [ ]:
A = torch.randn(4, 4, dtype=DTYPE)

x0 = A[:, 0].numpy()
beta, tau, v_tail = householder_vector(x0)
v = np.concatenate([[1.0], v_tail])
H1 = np.eye(4) - tau * np.outer(v, v)
A1 = H1 @ A.numpy()

show_row([A, A1], ["A", "H₁ A   (column 0 zeroed below diagonal)"])
print("First column of H1 A:", A1[:, 0], " <- only the top entry survives")

## 6. The full unblocked algorithm

Repeat on the trailing submatrix: column 0 of all rows, then column 1 of rows
$1{:}$, then column 2 of rows $2{:}$, and so on. Each step only touches the
bottom-right block that hasn't been triangularized yet. After $n{-}1$ steps the
strictly-lower triangle is zero — that's $R$.

Below we implement it ourselves and animate $R$ emerging, step by step.

In [ ]:
def my_qr(A):
    """Unblocked Householder QR, returning (R, list_of_(beta, tau, v_full))."""
    A = np.array(A.numpy() if hasattr(A, "numpy") else A, dtype=float)
    n = A.shape[0]
    reflectors = []
    for k in range(n):
        beta, tau, v_tail = householder_vector(A[k:, k])
        v = np.concatenate([[1.0], v_tail])         # length n-k
        # Apply H_k = I - tau v v^T to the trailing block A[k:, k:]
        A[k:, k:] -= tau * np.outer(v, v @ A[k:, k:])
        A[k:, k] = 0.0
        A[k, k] = beta
        reflectors.append((beta, tau, v))
    return A, reflectors


A = torch.randn(5, 5, dtype=DTYPE)

# Animate: snapshot the matrix after each column is cleared.
snaps, work = [A.numpy().copy()], A.numpy().copy()
n = work.shape[0]
for k in range(n):
    beta, tau, v_tail = householder_vector(work[k:, k])
    v = np.concatenate([[1.0], v_tail])
    work[k:, k:] -= tau * np.outer(v, v @ work[k:, k:])
    work[k:, k] = 0.0; work[k, k] = beta
    snaps.append(work.copy())

show_row(snaps, ["A"] + [f"after col {k}" for k in range(n)])

In [ ]:
# Does our R match a trusted QR? (R is unique up to signs of rows.)
R_mine, _ = my_qr(A)
_, R_ref = torch.linalg.qr(A)
print("Same |R| as torch.linalg.qr? ",
      np.allclose(np.abs(R_mine), np.abs(R_ref.numpy()), atol=1e-9))

## 7. The compact storage layout (the `geqrf` contract)

`torch.geqrf(A)` returns `(H, tau)` and packs everything into one matrix `H`
the same size as `A`, plus a vector `tau`:

- **Upper triangle of `H`** (including the diagonal) = $R$.
- **Strictly-lower triangle of `H`** = the reflector tails $v_{2:}$ for each
  column (the implicit `1` on the diagonal is *not* stored — it's understood).
- **`tau[k]`** = the coefficient $\tau_k$ for reflector $k$.

This is the storage layout your kernels must produce. Nothing is wasted: the
zeros that QR creates below the diagonal are reused to store the reflectors that
created them.

**An important subtlety:** the repo's reference and LAPACK's `geqrf` do **not**
produce the same `H`. A QR factorization is unique only *up to the signs of
$R$'s rows* — each algorithm is free to pick $\beta = +\lVert x\rVert$ or
$-\lVert x\rVert$ — so their `H` and `tau` legitimately differ (you'll see `tau`
gaps as large as 2.0). "Correct" therefore means **both reconstruct $A$**, not
that they match byte-for-byte. That's exactly what `tests/test_householder.py`
and the competition checker `local_eval.py` verify, and what `qr_errors` checks
below.

In [ ]:
A = torch.randn(5, 5, dtype=DTYPE)

H_geqrf, tau_geqrf = torch.geqrf(A)
H_ref, tau_ref = compact_householder_qr(A)   # the repo's reference implementation

# The repo's H and LAPACK's H are NOT identical -- they pick opposite signs for
# beta on some columns, so tau can differ by as much as 2.0. That is fine.
print("entrywise identical to geqrf?  ", torch.allclose(H_ref, H_geqrf, atol=1e-9))
print("max |tau_ref - tau_geqrf| =    ", round((tau_ref - tau_geqrf).abs().max().item(), 4))
print()
print("repo reference  recon/orth error: %.1e / %.1e" % qr_errors(A, H_ref, tau_ref))
print("torch.geqrf     recon/orth error: %.1e / %.1e" % qr_errors(A, H_geqrf, tau_geqrf))
print("-> both reconstruct A to machine precision, so BOTH are correct QR.")

# Visualize the packing: split H into its R part and its V-tail part.
H = H_geqrf.numpy()
R_part = np.triu(H)
V_part = np.tril(H, -1)
show_row([H, R_part, V_part],
         ["packed H from geqrf", "upper triangle = R", "lower triangle = v tails"])
print("\ntau =", tau_geqrf.numpy())

## 8. Rebuilding $Q$ and verifying $A = QR$

$Q = H_1 H_2 \cdots H_n$. We never form the $H_k$ explicitly in practice — we
just apply each reflector $I - \tau_k v_k v_k^\top$ to an identity matrix. We
unpack $v_k$ straight out of the compact `H` (remembering the implicit 1 on the
diagonal).

In [ ]:
def reconstruct_Q(H, tau):
    H = np.asarray(H, dtype=float)
    tau = np.asarray(tau, dtype=float)
    n = H.shape[0]
    Q = np.eye(n)
    for k in reversed(range(n)):          # apply in reverse so Q = H_1...H_n
        v = np.zeros(n)
        v[k] = 1.0
        v[k + 1:] = H[k + 1:, k]          # the stored tail
        Q -= tau[k] * np.outer(v, v @ Q)
    return Q


H, tau = (t.numpy() for t in torch.geqrf(A))
Q = reconstruct_Q(H, tau)
R = np.triu(H)

show_row([Q, R, Q @ R, A], ["Q (rebuilt)", "R", "Q @ R", "A (original)"])
print("Q orthogonal?  ", np.allclose(Q.T @ Q, np.eye(A.shape[0]), atol=1e-9))
print("Q @ R == A ?   ", np.allclose(Q @ R, A.numpy(), atol=1e-9))

## 9. Why GPUs block it — the compact-WY form

The unblocked loop applies reflectors one at a time, each a rank-1 update
($\mathcal{O}(n^2)$ work but memory-bound, lots of tiny operations). GPUs hate
that: they want big, dense matrix multiplies that saturate the tensor cores.

The fix is the **compact-WY representation**. A whole panel of $b$ consecutive
reflectors can be written as a *single* block reflector

$$H_1 H_2 \cdots H_b = I - V\,T\,V^\top$$

where $V$ stacks the reflector vectors (lower-trapezoidal, $b$ columns) and $T$
is a small $b\times b$ upper-triangular matrix. Applying this to the trailing
matrix becomes three big `bmm`s — exactly what `_apply_block_reflector_left`
does in the repo, and what your CUDA/Triton kernels are built around.

This is the bridge from "textbook QR" to "the thing being benchmarked in
`local_benchmark.py`". The panel factorization is still scalar Householder; only
the *trailing update* is blocked.

In [ ]:
A = torch.randn(2, 64, 64, dtype=DTYPE)   # a small *batch* of 64x64 matrices

H_block, tau_block = blocked_compact_householder_qr(A, block_size=16)
H_unblk, tau_unblk = compact_householder_qr(A)

# Blocked and unblocked share the SAME sign convention, so they match exactly:
print("blocked == unblocked (exact)?", torch.allclose(H_block, H_unblk, atol=1e-8))

# vs geqrf we compare by *validity*, not entrywise (sign freedom, see section 7):
print("blocked  recon/orth error: %.1e / %.1e" % qr_errors(A, H_block, tau_block))
print("geqrf    recon/orth error: %.1e / %.1e" % qr_errors(A, *torch.geqrf(A)))
print("\nSame factorization, different memory-access pattern — that's the whole game.")

### Where this lives in the repo

- `qr_practice/householder.py` — `compact_householder_qr` (sec 6) and
  `blocked_compact_householder_qr` (sec 9), the readable references.
- `baselines/` — `torch.geqrf` wrapped for profiling.
- `task.py` / `submission.py` — the competition contract you implement.
- `inputs.md` / `design.md` — which input shapes and edge cases (rank-deficient,
  clustered scales, mixed batches) force the *robust* path instead of a fast one.
- `kernels/` — the CUDA/Triton kernels that must reproduce exactly the `(H, tau)`
  you now understand.

## 10. 🔬 Playground — break it yourself

Things worth trying (these mirror the edge cases in `inputs.md`):

1. **Ill-conditioned input.** Build $A$ with a huge condition number and watch
   how stable the orthogonal factorization stays vs. naive Gaussian elimination.
2. **Rank-deficient column.** Make column 2 a copy of column 1. What $\tau$ does
   that produce? (Hint: the `norm == 0` / near-zero path.)
3. **Flip the sign convention** in `householder_vector` (use `+sign` instead of
   `-sign`) and measure how the error grows for nearly-axis-aligned columns —
   this is the cancellation the repo's sign choice avoids.
4. **Tune the block size** in section 9 and confirm correctness holds across
   `block_size in {1, 4, 16, 32}` — exactly the sweep `EXPERIMENTS.md` describes.

In [ ]:
# 1. Ill-conditioned: large spread of singular values, but QR stays orthogonal.
n = 6
U, _ = torch.linalg.qr(torch.randn(n, n, dtype=DTYPE))
V, _ = torch.linalg.qr(torch.randn(n, n, dtype=DTYPE))
svals = torch.logspace(0, 10, n, dtype=DTYPE)        # condition number ~ 1e10
A = U @ torch.diag(svals) @ V.T

H, tau = torch.geqrf(A)
Q = reconstruct_Q(H.numpy(), tau.numpy())
print("condition number ~ %.1e" % (svals.max() / svals.min()))
print("Q still orthogonal to 1e-9? ",
      np.allclose(Q.T @ Q, np.eye(n), atol=1e-9))
print("\nNow it's your turn — edit this cell.")

In [ ]:
# 2. Rank-deficient: column 2 duplicates column 1.
A = torch.randn(5, 5, dtype=DTYPE)
A[:, 2] = A[:, 1]
H, tau = torch.geqrf(A)
print("tau =", tau.numpy())
print("R diagonal =", np.diag(np.triu(H.numpy())), " <- look for the ~0 pivot")